In [ ]:
import arcpy

# --- Portal bilgileri ---
portal_url = "https://geo.ipa.istanbul/portal/"
username = "username"
password = "password"

# --- Portal'a giriş ---
arcpy.SignInToPortal(portal_url, username, password)

# --- Feature Service URL (REST, item linki değil) ---
feature_service_url = "https://geo.ipa.istanbul/server/rest/services/Hosted/detayli_ak/FeatureServer/0"

### --- Feature Layer oluştur ---
layer_name = "detayli_ak_lyr"
arcpy.management.MakeFeatureLayer(feature_service_url, layer_name)

# --- Basit kontrol ---
print("Portal:", arcpy.GetActivePortalURL())
print("Layer var mı?:", arcpy.Exists(layer_name))
print("Kayıt sayısı:", arcpy.management.GetCount(layer_name)[0])


Portal: https://geo.ipa.istanbul/portal/
Layer var mı?: True
Kayıt sayısı: 210636


In [2]:
import csv
from collections import Counter

# --- Katmanlar ---
source_layer = "detayli_ak_lyr"
filtered_layer = "detayli_ak_filtered"

# --- CSV yolu ---
csv_path = r"C:\Users\harun.korucu\Desktop\veri_temin_harun\scripts\domains.csv"

# --- CSV'den domain sözlüğü (kod -> açıklama) ---
domain_lookup = {}

with open(csv_path, newline='', encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        domain_lookup[str(row["Kod"])] = row["Adi"]

# --- Anlamlı veri filtresi ---
where_clause = "alt_kullanim IS NOT NULL"

arcpy.management.MakeFeatureLayer(
    source_layer,
    filtered_layer,
    where_clause
)

# --- Toplam kayıt ---
total = int(arcpy.management.GetCount(filtered_layer)[0])

# --- Frekans analizi ---
values = []
with arcpy.da.SearchCursor(filtered_layer, ["alt_kullanim"]) as cursor:
    for row in cursor:
        values.append(str(row[0]))  # kodlar string'e çevrildi

freq = Counter(values)

# --- ÇIKTI ---
print("ANLAMLI KAYIT SAYISI:", total)
print("\nALT_KULLANIM DAĞILIMI (açıklama / adet / %):\n")

for kod, adet in freq.most_common():
    aciklama = domain_lookup.get(kod, f"Bilinmeyen ({kod})")
    print(f"{aciklama} : {adet}  |  %{(adet/total)*100:.2f}")


ANLAMLI KAYIT SAYISI: 168525

ALT_KULLANIM DAĞILIMI (açıklama / adet / %):

KENTSEL KONUT ALANI : 58303  |  %34.60
TİCARET-KONUT ALANI : 28963  |  %17.19
REFÜJ/ŞEV-YAMAÇ/ÇAYIR : 18252  |  %10.83
YAPILAŞMAMIŞ ALAN : 12185  |  %7.23
TİCARET ALANI : 7867  |  %4.67
SANAYİ ALANI : 4183  |  %2.48
PARK : 3691  |  %2.19
İNŞAAT ALANI : 2664  |  %1.58
TARIM ALANI : 2164  |  %1.28
CAMİ/MESCİT ALANI : 2079  |  %1.23
SANAYİ-TİCARET ALANI : 1546  |  %0.92
ORMAN ALANI : 1457  |  %0.86
KIRSAL KONUT ALANI : 1316  |  %0.78
SANAYİ-KONUT ALANI : 1306  |  %0.77
AÇIK OTOPARK : 1278  |  %0.76
AĞAÇLIK ALAN : 1199  |  %0.71
SANAYİ-TİCARET-KONUT ALANI : 1103  |  %0.65
AKARSU/DERE : 985  |  %0.58
ANAOKULU ALANI : 948  |  %0.56
KAMU KURUMU (YEREL İDARE) : 925  |  %0.55
YAYA BÖLGESİ : 721  |  %0.43
İLKOKUL ALANI : 719  |  %0.43
ORGANİZE SANAYİ BÖLGESİ : 711  |  %0.42
KAMU KURUMU (MERKEZİ İDARE) : 707  |  %0.42
LİSE ALANI : 681  |  %0.40
DEPOLAMA VE LOJİSTİK TESİS ALANI : 644  |  %0.38
TURİZM ALANI (KONAKLAMA) : 63

In [3]:
import os
# Kaynak layer (zaten hazır)
source_layer = "detayli_ak_lyr"

# GDB'nin oluşturulacağı klasör
workspace_folder = r"C:\Users\harun.korucu\Desktop\veri_temin_harun"

# GDB adı
gdb_name = "detayli_ak_local.gdb"
gdb_path = os.path.join(workspace_folder, gdb_name)

# Eğer GDB yoksa oluştur
if not arcpy.Exists(gdb_path):
    arcpy.management.CreateFileGDB(workspace_folder, gdb_name)

# Feature class adı
output_fc = os.path.join(gdb_path, "detayli_ak_filtered")

# Feature class olarak kopyala
arcpy.management.CopyFeatures(source_layer, output_fc)

print("Lokal GDB oluşturuldu:", gdb_path)
print("Feature class yazıldı:", output_fc)
print("Kayıt sayısı:", arcpy.management.GetCount(output_fc)[0])

Lokal GDB oluşturuldu: C:\Users\harun.korucu\Desktop\veri_temin_harun\detayli_ak_local.gdb
Feature class yazıldı: C:\Users\harun.korucu\Desktop\veri_temin_harun\detayli_ak_local.gdb\detayli_ak_filtered
Kayıt sayısı: 210646


In [4]:
from datetime import datetime
from collections import defaultdict
import arcpy
import os

# --- Workspace ---
gdb_path = r"C:\Users\harun.korucu\Desktop\veri_temin_harun\detayli_ak_local.gdb"
arcpy.env.workspace = gdb_path
arcpy.env.overwriteOutput = True

# --- Kaynak layer ---
fc = "detayli_ak_lyr"

# --- Tarih ---
today = datetime.now().strftime("%Y%m%d")

# =====================================================
# 🔥 1. HAM VERİYİ TARİHLİ KAYDET (GEOMETRY SAFE)
# =====================================================
out_raw_fc = os.path.join(gdb_path, f"detayli_ak_{today}")

arcpy.management.CopyFeatures(fc, out_raw_fc)

print("Ham veri kaydedildi:", out_raw_fc)
print("Kayıt:", arcpy.management.GetCount(out_raw_fc)[0])

# =====================================================
# 🔍 2. REFERANS İLÇE ANALİZİ
# =====================================================

# Field adlarını güvenli al
ilce_field = [f.name for f in arcpy.ListFields(fc) if f.name.lower() == "ilce"][0]
alt_field = [f.name for f in arcpy.ListFields(fc) if f.name.lower() == "alt_kullanim"][0]

ilce_stats = defaultdict(lambda: {"total": 0, "null_alt": 0})

with arcpy.da.SearchCursor(fc, [ilce_field, alt_field]) as cursor:
    for ilce, alt in cursor:
        ilce_stats[ilce]["total"] += 1
        if alt in (None, "", " "):
            ilce_stats[ilce]["null_alt"] += 1

referans_ilceler = [
    ilce for ilce, stats in ilce_stats.items()
    if stats["null_alt"] == 0
]

print("Referans ilçe sayısı:", len(referans_ilceler))

if not referans_ilceler:
    raise Exception("Referans ilçe bulunamadı!")

# =====================================================
# 🧠 3. SEÇ + EXPORT (GEOMETRY SAFE)
# =====================================================

out_ref_fc = f"detayli_ak_referans_ilceler_{today}"

arcpy.management.MakeFeatureLayer(fc, "ref_layer")
arcpy.management.SelectLayerByAttribute("ref_layer", "CLEAR_SELECTION")

def format_sql(val):
    return f"'{val}'" if isinstance(val, str) else str(val)

ilce_sql = ",".join([format_sql(i) for i in referans_ilceler])
where_clause = f"{ilce_field} IN ({ilce_sql})"

arcpy.management.SelectLayerByAttribute(
    "ref_layer",
    "NEW_SELECTION",
    where_clause
)

arcpy.management.CopyFeatures("ref_layer", out_ref_fc)

# =====================================================
# 📊 SONUÇ
# =====================================================

print("Referans FC:", out_ref_fc)
print("Kayıt:", arcpy.management.GetCount(out_ref_fc)[0])

desc = arcpy.Describe(out_ref_fc)
print("Geometry:", desc.shapeType)
print("Spatial Ref:", desc.spatialReference.name)

Ham veri kaydedildi: C:\Users\harun.korucu\Desktop\veri_temin_harun\detayli_ak_local.gdb\detayli_ak_20260325
Kayıt: 210655
Referans ilçe sayısı: 7
Referans FC: detayli_ak_referans_ilceler_20260325
Kayıt: 34500
Geometry: Polygon
Spatial Ref: WGS_1984_Web_Mercator_Auxiliary_Sphere
